In [ ]:
!pip install -q -U \
    transformers accelerate peft \
    bitsandbytes sentencepiece protobuf huggingface_hub \
    --break-system-packages

# After this cell finishes: Runtime → Restart session

In [ ]:
# Cell 1 — Fix torchaudio + torchvision CUDA mismatch
!pip uninstall -y torchaudio torchvision torchao 2>/dev/null

# Reinstall matching your torch's CUDA 13.0
!pip install torchaudio torchvision --index-url https://download.pytorch.org/whl/cu130 --break-system-packages

# Restart runtime after this

In [ ]:
import os
os.environ["USE_TF"] = "0"

import torch, torchaudio, torchvision
print(f"torch:       {torch.__version__}")
print(f"torchaudio:  {torchaudio.__version__}")
print(f"torchvision: {torchvision.__version__}")

import peft
print(f"peft:        {peft.__version__}")

In [ ]:
import os
os.environ["USE_TF"] = "0"                    # prevent TensorFlow from loading
os.environ["UNSLOTH_DISABLE_TORCHAO"] = "1"   # just in case

# ============================================================
# EDIT THESE TWO VALUES
# ============================================================

# Your HF repo containing the LoRA adapter
ADAPTER_REPO = "skylord/gemma-3n-kcc-sample100k-step6300"

# The base model your adapter was trained on.
# Check adapter_config.json on your HF repo — look at "base_model_name_or_path"
BASE_MODEL = "google/gemma-3n-E2B-it"

# Quantization type for final GGUF (options: Q4_K_M, Q5_K_M, Q6_K, Q8_0)
QUANT_TYPE = "Q4_K_M"

# ============================================================
# DON'T EDIT BELOW
# ============================================================
MERGED_DIR = "/content/merged"
LLAMA_CPP_DIR = "/content/llama.cpp"
BF16_GGUF = "/content/gemma-3n-e2b-it.BF16.gguf"
FINAL_GGUF = f"/content/gemma-3n-e2b-it.{QUANT_TYPE}.gguf"

print("Config loaded.")
print(f"  Adapter:    {ADAPTER_REPO}")
print(f"  Base model: {BASE_MODEL}")
print(f"  Quant:      {QUANT_TYPE}")

In [ ]:

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

In [ ]:
import torch

# Try the specific class first, fall back to AutoModel
try:
    from transformers import Gemma3nForConditionalGeneration as ModelClass
    print("Using Gemma3nForConditionalGeneration")
except ImportError:
    from transformers import AutoModel as ModelClass
    print("Using AutoModel (fallback)")

In [ ]:
from peft import PeftModel

print(f"\n[1/3] Loading base model: {BASE_MODEL}")
print("      This downloads ~11 GB on first run...")
base_model = ModelClass.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="cpu",          # CPU merge — safe and uses system RAM
    low_cpu_mem_usage=True,
    # token=HF_TOKEN            # Add this line for authentication
)
print("      Base model loaded.")

In [ ]:
print(f"\n[2/3] Applying LoRA adapter: {ADAPTER_REPO}")
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
print("      Adapter applied.")

print("\n[3/3] Merging weights...")
model = model.merge_and_unload()
print("      Merge complete.")


In [ ]:
# Save to local disk
print(f"\nSaving merged model to {MERGED_DIR}...")
model.save_pretrained(MERGED_DIR)

In [ ]:
# Save tokenizer/processor
from transformers import AutoTokenizer
try:
    from transformers import AutoProcessor
    processor = AutoProcessor.from_pretrained(ADAPTER_REPO)
    processor.save_pretrained(MERGED_DIR)
    print("Processor saved.")
except Exception:
    pass

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_REPO)
tokenizer.save_pretrained(MERGED_DIR)
print("Tokenizer saved.")


In [ ]:
# Free memory
del model, base_model
import gc; gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("\n✅ Merge complete!")

In [ ]:
import os

print("Merged model contents:")
print("-" * 60)
total = 0
for f in sorted(os.listdir(MERGED_DIR)):
    path = os.path.join(MERGED_DIR, f)
    size = os.path.getsize(path)
    total += size
    marker = " ⚠️ EMPTY!" if size == 0 else ""
    print(f"  {size:>14,} bytes  {f}{marker}")

print("-" * 60)
print(f"  Total: {total / 1e9:.2f} GB")


In [ ]:

# Check for empty safetensors
import glob
shards = glob.glob(f"{MERGED_DIR}/*.safetensors")
empty = [s for s in shards if os.path.getsize(s) == 0]
if empty:
    print(f"\n❌ ERROR: Empty shards found: {empty}")
    print("   The merge failed silently. Re-run Cell 4.")
elif not shards:
    print(f"\n❌ ERROR: No .safetensors files found in {MERGED_DIR}")
else:
    print(f"\n✅ All {len(shards)} shards look good. Proceed to conversion.")

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp {LLAMA_CPP_DIR}
!pip install -r {LLAMA_CPP_DIR}/requirements.txt --break-system-packages
print("\n✅ llama.cpp ready.")

In [ ]:
import os
os.environ["USE_TF"] = "0"                    # prevent TensorFlow from loading
os.environ["UNSLOTH_DISABLE_TORCHAO"] = "1"   # just in case

# ============================================================
# EDIT THESE TWO VALUES
# ============================================================

# Your HF repo containing the LoRA adapter
ADAPTER_REPO = "skylord/gemma-3n-kcc-sample100k-step6300"

# The base model your adapter was trained on.
# Check adapter_config.json on your HF repo — look at "base_model_name_or_path"
BASE_MODEL = "google/gemma-3n-E2B-it"

# Quantization type for final GGUF (options: Q4_K_M, Q5_K_M, Q6_K, Q8_0)
QUANT_TYPE = "Q4_K_M"

# ============================================================
# DON'T EDIT BELOW
# ============================================================
MERGED_DIR = "/content/merged"
LLAMA_CPP_DIR = "/content/llama.cpp"
BF16_GGUF = "/content/gemma-3n-e2b-it.BF16.gguf"
FINAL_GGUF = f"/content/gemma-3n-e2b-it.{QUANT_TYPE}.gguf"

print("Config loaded.")
print(f"  Adapter:    {ADAPTER_REPO}")
print(f"  Base model: {BASE_MODEL}")
print(f"  Quant:      {QUANT_TYPE}")

In [ ]:
!python {LLAMA_CPP_DIR}/convert_hf_to_gguf.py \
    {MERGED_DIR} \
    --outfile {BF16_GGUF} \
    --outtype bf16

import os
if os.path.exists(BF16_GGUF) and os.path.getsize(BF16_GGUF) > 0:
    size_gb = os.path.getsize(BF16_GGUF) / 1e9
    print(f"\n✅ BF16 GGUF created: {size_gb:.2f} GB")
else:
    print("\n❌ Conversion failed. Check error above.")

In [ ]:
# Download prebuilt llama.cpp binaries
!wget -q https://github.com/ggml-org/llama.cpp/releases/download/b9672/llama-b9672-bin-ubuntu-x64.tar.gz -O /content/llama-bin.tar.gz
!mkdir -p /content/llama-bin && tar xzf /content/llama-bin.tar.gz -C /content/llama-bin

# Find the quantize binary
import glob
quantize_bin = glob.glob("/content/llama-bin/**/llama-quantize", recursive=True)
if not quantize_bin:
    # Try building from source as fallback
    print("Prebuilt binary not found, building from source...")
    !cd {LLAMA_CPP_DIR} && cmake -B build && cmake --build build --target llama-quantize -j$(nproc)
    quantize_bin = [f"{LLAMA_CPP_DIR}/build/bin/llama-quantize"]

QUANTIZE = quantize_bin[0]
print(f"Using: {QUANTIZE}")

!{QUANTIZE} {BF16_GGUF} {FINAL_GGUF} {QUANT_TYPE}

import os
if os.path.exists(FINAL_GGUF) and os.path.getsize(FINAL_GGUF) > 0:
    size_gb = os.path.getsize(FINAL_GGUF) / 1e9
    print(f"\n✅ Final GGUF: {FINAL_GGUF} ({size_gb:.2f} GB)")
else:
    print("\n❌ Quantization failed. Check error above.")

In [ ]:
from google.colab import drive
drive.mount('/gdrive')

DRIVE_DEST = "/gdrive/MyDrive/models/"
!mkdir -p "{DRIVE_DEST}"
!cp -v {FINAL_GGUF} "{DRIVE_DEST}"

# Verify
import os
drive_file = os.path.join(DRIVE_DEST, os.path.basename(FINAL_GGUF))
local_size = os.path.getsize(FINAL_GGUF)
drive_size = os.path.getsize(drive_file)
if local_size == drive_size:
    print(f"\n✅ Copied to Drive. Size matches: {drive_size / 1e9:.2f} GB")
else:
    print(f"\n⚠️ Size mismatch! Local: {local_size}, Drive: {drive_size}")
    print("   Drive may still be syncing. Wait a minute and re-check.")

In [ ]:
# from google.colab import files
# files.download(FINAL_GGUF)

In [ ]:
from huggingface_hub import HfApi, login

# ============================================================
# EDIT THESE VALUES
# ============================================================
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
GGUF_REPO = "skylord/gemma-3n-kcc-sample100k-gguf"  # target repo name
# ============================================================

login(token=HF_TOKEN)
api = HfApi()

In [ ]:
# help(api.create_repo)

In [ ]:
# Create the repo (skip if it already exists)
api.create_repo(
    token=HF_TOKEN,
    repo_id=GGUF_REPO,
    repo_type="model",
    exist_ok=True,
)


In [ ]:
GGUF_REPO

In [ ]:
# Upload the quantized GGUF
import os
gguf_filename = os.path.basename(FINAL_GGUF)
print(f"Uploading {gguf_filename} ({os.path.getsize(FINAL_GGUF) / 1e9:.2f} GB)...")
print("This may take a few minutes...")

api.upload_file(
    token=HF_TOKEN,
    path_or_fileobj=FINAL_GGUF,
    path_in_repo=gguf_filename,
    repo_id=GGUF_REPO,
)
print(f"\n✅ Uploaded to https://huggingface.co/{GGUF_REPO}")

In [ ]:
# Optionally upload the BF16 GGUF too (larger, but useful as a base for other quants)
UPLOAD_BF16_TOO = False  # Change to True if you want both

if UPLOAD_BF16_TOO and os.path.exists(BF16_GGUF):
    bf16_filename = os.path.basename(BF16_GGUF)
    print(f"\nUploading {bf16_filename} ({os.path.getsize(BF16_GGUF) / 1e9:.2f} GB)...")
    api.upload_file(
        token=HF_TOKEN,
        path_or_fileobj=BF16_GGUF,
        path_in_repo=bf16_filename,
        repo_id=GGUF_REPO,
    )
    print(f"✅ BF16 uploaded too.")

print(f"\n🎉 Done! View your model at: https://huggingface.co/{GGUF_REPO}")